# Finetune Embeddings


Finetuning embedding models often heavily improves the performance of the model on your use case, because each task requires a different notion of similarity. For example, given news articles:
- “Apple launches the new iPad”
- “NVIDIA is gearing up for the next GPU generation”
Then the following use cases, we may have different notions of similarity:
- a model for classification of news articles as Economy, Sports, Technology, Politics, etc., should produce similar embeddings for these texts.
- a model for semantic textual similarity should produce dissimilar embeddings for these texts, as they have different meanings.
- a model for semantic search would not need a notion for similarity between two documents, as it should only compare queries and documents.

# Preparing the training and validation dataset

In [ ]:
!pip -q install langchain-community langchain-text-splitters openai
!pip -q install pypdf

In [ ]:
import os
import json
from typing import List, Dict
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from openai import OpenAI

class PDFProcessor:
    """Process PDF documents into text chunks."""

    def __init__(self, pdf_path: str, chunk_size: int = 1000,
                 chunk_overlap: int = 200):
        """
        Initialize PDF processor.

        Args:
            pdf_path: Path to PDF file
            chunk_size: Size of each text chunk
            chunk_overlap: Overlap between chunks
        """
        self.pdf_path = pdf_path
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

    def process(self) -> List[str]:
        """
        Load PDF and split into chunks.

        Returns:
            List of text chunks
        """
        # Load PDF
        loader = PyPDFLoader(self.pdf_path)
        documents = loader.load()

        # Split into chunks
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
            length_function=len,
            separators=["\n\n", "\n", " ", ""]
        )

        chunks = text_splitter.split_documents(documents)
        return [chunk.page_content for chunk in chunks]


In [ ]:
class QAPairGenerator:
    """Generic QA pair generator that works with text chunks from any source."""

    def __init__(self, openai_api_key: str, model: str = "gpt-4o-mini"):
        """
        Initialize the QA pair generator.

        Args:
            openai_api_key: Your OpenAI API key
            model: OpenAI model to use (default: gpt-4o-mini for cost efficiency)
        """
        self.client = OpenAI(api_key=openai_api_key, base_url=endpoint)
        self.model = model

    def generate_questions(self, context: str, num_questions: int = 3) -> List[str]:
        """
        Generate questions based on a given context using OpenAI.

        Args:
            context: Text context to generate questions from
            num_questions: Number of questions to generate per chunk

        Returns:
            List of generated questions
        """
        prompt = f"""Based on the following text, generate {num_questions} diverse questions that can be answered using the information in the text.

The questions should:
- Be clear and specific
- Cover different aspects of the content
- Be answerable from the given context
- Vary in complexity (some simple, some requiring deeper understanding)

Text:
{context}

Return only the questions, one per line, without numbering or additional text."""

        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": "You are a helpful assistant that generates high-quality questions for training embedding models."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.7,
                max_tokens=500
            )

            questions_text = response.choices[0].message.content.strip()
            questions = [q.strip() for q in questions_text.split('\n') if q.strip()]
            return questions

        except Exception as e:
            print(f"Error generating questions: {e}")
            return []

    def create_batch_file(self, text_chunks: List[str],
                         num_questions: int = 3,
                         output_file: str = "batch_requests.jsonl") -> str:
        """
        Create a JSONL file for OpenAI Batch API.

        Args:
            text_chunks: List of text chunks
            num_questions: Number of questions per chunk
            output_file: Path to output JSONL file

        Returns:
            Path to the created batch file
        """
        with open(output_file, 'w', encoding='utf-8') as f:
            for i, chunk in enumerate(text_chunks):
                prompt = f"""Based on the following text, generate {num_questions} diverse questions that can be answered using the information in the text.

The questions should:
- Be clear and specific
- Cover different aspects of the content
- Be answerable from the given context
- Vary in complexity (some simple, some requiring deeper understanding)

Text:
{chunk}

Return only the questions, one per line, without numbering or additional text."""

                request = {
                    "custom_id": f"request-{i}",
                    "method": "POST",
                    "url": "/v1/chat/completions",
                    "body": {
                        "model": self.model,
                        "messages": [
                            {"role": "system", "content": "You are a helpful assistant that generates high-quality questions for training embedding models."},
                            {"role": "user", "content": prompt}
                        ],
                        "temperature": 0.7,
                        "max_tokens": 500
                    }
                }
                f.write(json.dumps(request) + '\n')

        print(f"Created batch file: {output_file}")
        return output_file

    def submit_batch(self, batch_file_path: str) -> str:
        """
        Submit batch file to OpenAI Batch API.

        Args:
            batch_file_path: Path to the batch JSONL file

        Returns:
            Batch ID
        """
        # Upload the batch file
        with open(batch_file_path, 'rb') as f:
            batch_input_file = self.client.files.create(
                file=f,
                purpose="batch"
            )

        print(f"Uploaded batch file: {batch_input_file.id}")

        # Create the batch
        batch = self.client.batches.create(
            input_file_id=batch_input_file.id,
            endpoint="/v1/chat/completions",
            completion_window="24h"
        )

        print(f"Batch created: {batch.id}")
        print(f"Status: {batch.status}")
        return batch.id

    def check_batch_status(self, batch_id: str) -> Dict:
        """
        Check the status of a batch job.

        Args:
            batch_id: The batch ID

        Returns:
            Batch status information
        """
        batch = self.client.batches.retrieve(batch_id)
        return {
            "id": batch.id,
            "status": batch.status,
            "created_at": batch.created_at,
            "completed_at": batch.completed_at,
            "failed_at": batch.failed_at,
            "request_counts": {
                "total": batch.request_counts.total,
                "completed": batch.request_counts.completed,
                "failed": batch.request_counts.failed
            }
        }

    def retrieve_batch_results(self, batch_id: str,
                              output_file: str = "batch_results.jsonl") -> str:
        """
        Retrieve results from a completed batch.

        Args:
            batch_id: The batch ID
            output_file: Path to save results

        Returns:
            Path to the results file
        """
        batch = self.client.batches.retrieve(batch_id)

        if batch.status != "completed":
            raise Exception(f"Batch not completed. Current status: {batch.status}")

        # Download the result file
        result_file_id = batch.output_file_id
        result = self.client.files.content(result_file_id)

        with open(output_file, 'wb') as f:
            f.write(result.content)

        print(f"Results saved to: {output_file}")
        return output_file

    def parse_batch_results(self, results_file: str,
                           text_chunks: List[str]) -> List[Dict]:
        """
        Parse batch results and create QA pairs.

        Args:
            results_file: Path to batch results JSONL file
            text_chunks: Original text chunks (to match with results)

        Returns:
            List of QA pairs
        """
        qa_pairs = []

        # Read and parse results
        with open(results_file, 'r', encoding='utf-8') as f:
            for line in f:
                result = json.loads(line)
                custom_id = result["custom_id"]
                chunk_idx = int(custom_id.split('-')[1])

                if result["response"]["status_code"] == 200:
                    content = result["response"]["body"]["choices"][0]["message"]["content"]
                    questions = [q.strip() for q in content.strip().split('\n') if q.strip()]

                    for question in questions:
                        qa_pairs.append({
                            "query": question,
                            "corpus": text_chunks[chunk_idx],
                            "chunk_id": chunk_idx
                        })
                else:
                    print(f"Error in request {custom_id}: {result['response']}")

        print(f"Generated {len(qa_pairs)} QA pairs from batch results")
        return qa_pairs

    def generate_qa_pairs(self, text_chunks: List[str],
                         num_questions_per_chunk: int = 3,
                         min_chunk_length: int = 100,
                         use_batch_api: bool = False,
                         batch_file: str = "batch_requests.jsonl") -> List[Dict]:
        """
        Generate QA pairs from text chunks.

        Args:
            text_chunks: List of text chunks to process
            num_questions_per_chunk: Number of questions to generate per chunk
            min_chunk_length: Minimum chunk length to process
            use_batch_api: Whether to use Batch API (if True, returns batch_id instead)
            batch_file: Path for batch file if using Batch API

        Returns:
            List of QA pairs (or batch_id if use_batch_api=True)
        """
        # Filter out short chunks
        valid_chunks = []
        valid_indices = []

        for i, chunk in enumerate(text_chunks):
            if len(chunk.strip()) >= min_chunk_length:
                valid_chunks.append(chunk)
                valid_indices.append(i)
            else:
                print(f"Skipping chunk {i} (too short)")

        print(f"Processing {len(valid_chunks)} valid chunks...")

        if use_batch_api:
            # Create and submit batch
            self.create_batch_file(valid_chunks, num_questions_per_chunk, batch_file)
            batch_id = self.submit_batch(batch_file)
            print(f"\nBatch submitted! Use batch_id '{batch_id}' to check status and retrieve results.")
            print(f"Check status with: qa_generator.check_batch_status('{batch_id}')")
            print(f"Retrieve results with: qa_generator.retrieve_batch_results('{batch_id}')")
            return {"batch_id": batch_id, "valid_chunks": valid_chunks}
        else:
            # Use regular API (synchronous)
            qa_pairs = []
            for i, chunk in enumerate(valid_chunks):
                print(f"Processing chunk {i+1}/{len(valid_chunks)}...")
                questions = self.generate_questions(chunk, num_questions_per_chunk)

                for question in questions:
                    qa_pairs.append({
                        "query": question,
                        "corpus": chunk,
                        "chunk_id": valid_indices[i]
                    })

            print(f"Generated {len(qa_pairs)} QA pairs")
            return qa_pairs

    def save_to_json(self, qa_pairs: List[Dict], output_path: str):
        """Save QA pairs to JSON file."""
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(qa_pairs, f, indent=2, ensure_ascii=False)
        print(f"Saved QA pairs to {output_path}")

    def save_to_sentence_transformer_format(self, qa_pairs: List[Dict], output_path: str):
        """
        Save in format optimized for Sentence Transformers training.
        Format: List of (query, positive_passage) pairs
        """
        st_format = []
        for pair in qa_pairs:
            st_format.append({
                "query": pair["query"],
                "pos": [pair["corpus"]]  # Positive passage
            })

        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(st_format, f, indent=2, ensure_ascii=False)
        print(f"Saved Sentence Transformer format to {output_path}")


In [ ]:
!mkdir -p 'data/10k/'
!wget -q 'https://raw.githubusercontent.com/run-llama/llama_index/main/docs/examples/data/10k/uber_2021.pdf' -O 'data/10k/uber_2021.pdf'
!wget -q 'https://raw.githubusercontent.com/run-llama/llama_index/main/docs/examples/data/10k/lyft_2021.pdf' -O 'data/10k/lyft_2021.pdf'

In [ ]:
TRAIN_FILE = "./data/10k/lyft_2021.pdf"
VAL_FILE = "./data/10k/uber_2021.pdf"

TRAIN_CORPUS_FPATH = "./data/train_corpus.json"
VAL_CORPUS_FPATH = "./data/val_corpus.json"

TRAIN_CORPUS_ST_FPATH = "./data/train_corpus_st.json"
VAL_CORPUS_ST_FPATH = "./data/val_corpus_st.json"

In [ ]:
endpoint = "https://nypopenai2.cognitiveservices.azure.com/openai/v1/"
model_name = "gpt-4o-mini"
deployment_name = "gpt-4o-mini"
api_key = "<api key>"

In [ ]:
# Set your OpenAI API key
# api_key = os.getenv("OPENAI_API_KEY", "your-api-key-here")

# Initialize QA generator
qa_generator = QAPairGenerator(openai_api_key=api_key)

In [ ]:
## Create Train and Validation set

# Training set
train_pdf_processor = PDFProcessor(
    pdf_path=TRAIN_FILE,
    chunk_size=1000,
    chunk_overlap=200
)
train_pdf_chunks = train_pdf_processor.process()

# Submit batch job
train_batch_info = qa_generator.generate_qa_pairs(
    text_chunks=train_pdf_chunks,
    num_questions_per_chunk=2,
    use_batch_api=True
)

train_batch_id = train_batch_info["batch_id"]
train_valid_chunks = train_batch_info["valid_chunks"]

validation_pdf_processor = PDFProcessor(
    pdf_path=TRAIN_FILE,
    chunk_size=1000,
    chunk_overlap=200
)
validation_pdf_chunks = validation_pdf_processor.process()

# Submit batch job
validation_batch_info = qa_generator.generate_qa_pairs(
    text_chunks=validation_pdf_chunks,
    num_questions_per_chunk=2,
    use_batch_api=True
)

validation_batch_id = validation_batch_info["batch_id"]
validation_valid_chunks = validation_batch_info["valid_chunks"]


In [ ]:
# Check status (do this periodically until completed)
print("\nChecking batch status...")
train_batch_status = qa_generator.check_batch_status(train_batch_id)
validation_batch_status = qa_generator.check_batch_status(validation_batch_id)
print(f"Training batch status: {train_batch_status}\nValidation batch status: {validation_batch_status}")

In [ ]:
train_batch_completed = False
validation_batch_completed = False

while train_batch_completed is False or validation_batch_completed is False:
    if train_batch_completed is False:
        train_batch_status = qa_generator.check_batch_status(train_batch_id)
        if train_batch_status["status"] == "completed":
            print("train batch completed")
            train_batch_completed = True
        else:
            print(f"train batch status = {train_batch_status['status']}")
    if validation_batch_completed is False:
        validation_batch_status = qa_generator.check_batch_status(validation_batch_id)
        if validation_batch_status["status"] == "completed":
            print("validation batch completed")
            validation_batch_completed = True
        else:
            print(f"validation batch status = {validation_batch_status['status']}")

In [ ]:
# Once completed, retrieve and parse results
# Uncomment these lines after batch is completed (usually takes a few minutes to hours)
results_file = qa_generator.retrieve_batch_results(train_batch_id)
train_qa_pairs = qa_generator.parse_batch_results(results_file, train_valid_chunks)
qa_generator.save_to_json(train_qa_pairs, TRAIN_CORPUS_FPATH)
qa_generator.save_to_sentence_transformer_format(train_qa_pairs, TRAIN_CORPUS_ST_FPATH)

results_file = qa_generator.retrieve_batch_results(validation_batch_id)
validation_qa_pairs = qa_generator.parse_batch_results(results_file, validation_valid_chunks)
qa_generator.save_to_json(validation_qa_pairs, VAL_CORPUS_FPATH)
qa_generator.save_to_sentence_transformer_format(validation_qa_pairs, VAL_CORPUS_ST_FPATH)

In [ ]:
import json
from datasets import Dataset, DatasetDict

with open(TRAIN_CORPUS_ST_FPATH, 'r') as f:
    train_qa_pairs = json.load(f)
    data_dict = {
        "query": [item["query"] for item in train_qa_pairs],
        "pos": [item["pos"] for item in train_qa_pairs]
    }

    # Create the dataset
    train_dataset = Dataset.from_dict(data_dict)

with open(VAL_CORPUS_ST_FPATH, 'r') as f:
    validation_qa_pairs = json.load(f)
    data_dict = {
        "query": [item["query"] for item in validation_qa_pairs],
        "pos": [item["pos"] for item in validation_qa_pairs]
    }

    # Create the dataset
    validation_dataset = Dataset.from_dict(data_dict)

In [ ]:
dataset = DatasetDict({
    'train': train_dataset,
    'validation': validation_dataset
})

In [ ]:
dataset.push_to_hub("khengkok/annual-report-qa-dataset")